# ゼロ幅文字・制御文字・異体字セレクタを取り除く

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_string/invisible_chars.ipynb)

NLP クックブック 03-21 ／ 文字列処理と正規化

見た目は同じなのに文字列が一致しない原因になる、目に見えない文字を検出・除去したい。

In [1]:
import unicodedata

text = "自然​言語﻿処理 と　葛\U000e0100飾区\x07"
for c in text:
    cat = unicodedata.category(c)
    if cat[0] in "CZM":
        print(f"U+{ord(c):04X} {cat} {unicodedata.name(c, '(no name)')}")

U+200B Cf ZERO WIDTH SPACE
U+FEFF Cf ZERO WIDTH NO-BREAK SPACE
U+00A0 Zs NO-BREAK SPACE
U+3000 Zs IDEOGRAPHIC SPACE
U+E0100 Mn VARIATION SELECTOR-17
U+0007 Cc (no name)


In [2]:
def remove_invisible(text):
    out = []
    for c in text:
        cat = unicodedata.category(c)
        if cat in ("Cc", "Cf") and c not in "\n\t":   # 制御文字・書式文字 (ゼロ幅スペース, BOM など)
            continue
        if 0xFE00 <= ord(c) <= 0xFE0F or 0xE0100 <= ord(c) <= 0xE01EF:  # 異体字セレクタ
            continue
        out.append(" " if cat == "Zs" else c)          # 各種スペースを半角スペースに
    return "".join(out)

clean = remove_invisible(text)
print(repr(clean), len(text), "→", len(clean))

'自然言語処理 と 葛飾区' 16 → 12


- `Cf` を消すと絵文字の ZWJ (U+200D) も消えて家族の絵文字などが分解される。絵文字を残したいなら U+200D は除外する。